In [1]:
# ================= BLOCK 0c FINAL: verified-duplicate grouping + folds =================
import hashlib, json, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

APTOS = Path("/kaggle/input/competitions/aptos2019-blindness-detection")
OUT = Path("/kaggle/working/cgrl"); OUT.mkdir(parents=True, exist_ok=True)
cv2.setNumThreads(1); SEED = 2026
S = 128; R = 4; HAM = 10; NR_TH = 0.035; SS_TH = 0.90; CAP = 16

df = pd.read_csv(APTOS/"train.csv")
df["path"] = df.id_code.map(lambda x: str(APTOS/"train_images"/f"{x}.png"))
paths = df.path.tolist(); grades = df.diagnosis.values; N = len(paths)
print(f"APTOS raw: {N} | grades {np.bincount(grades,minlength=5).tolist()}", flush=True)

def fov(im):
    sm = cv2.GaussianBlur(im, (5,5), 0)                       # blur ONLY for the bbox mask
    _, th = cv2.threshold(sm, 10, 255, cv2.THRESH_BINARY)
    ys, xs = np.where(th > 0)
    return im[ys.min():ys.max()+1, xs.min():xs.max()+1] if len(xs) > 100 else im   # crisp slice

def phash64(im):
    g = cv2.resize(im, (32,32), interpolation=cv2.INTER_AREA).astype(np.float32)
    b = cv2.dct(g)[:8,:8].flatten(); bits = (b > np.median(b[1:])).astype(np.uint8); v = 0
    for x in bits: v = (v << 1) | int(x)
    return v

def feat(i):
    p = paths[i]
    try: buf = Path(p).read_bytes()
    except Exception: return i, None, None
    m = hashlib.md5(buf).hexdigest()
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if im is None: return i, m, None
    u = cv2.resize(fov(im), (S,S), interpolation=cv2.INTER_AREA)
    return i, m, u

MD5 = [None]*N; H = [None]*N; U = np.zeros((N,S,S), np.uint8)
with ThreadPoolExecutor(4) as ex:
    for i, m, u in tqdm(ex.map(feat, range(N)), total=N, desc="md5+FOV thumb", leave=False):
        MD5[i] = m
        if u is not None: U[i] = u; H[i] = phash64(u)

A = np.array([h or 0 for h in H], np.uint64)
pc = np.array([bin(i).count("1") for i in range(256)], np.uint8)
ci_, cj_ = [], []
for s in range(0, N, 512):
    e = min(s+512, N); x = A[s:e, None] ^ A[None, :]
    d = np.zeros(x.shape, np.uint8)
    for k in range(8): d += pc[((x >> np.uint64(8*k)) & np.uint64(255)).astype(np.uint8)]
    ii, jj = np.where(d <= HAM)
    for a_, b_ in zip(ii+s, jj):
        if a_ < b_ and H[a_] is not None and H[b_] is not None:
            ci_.append(int(a_)); cj_.append(int(b_))
ci = np.array(ci_, np.int32); cj = np.array(cj_, np.int32)
print(f"candidates (Hamming<={HAM}): {len(ci)}", flush=True)

def aligned(a, b, r=R):
    ac = a[r:-r, r:-r].astype(np.float32); best = (1e9, ac, ac)
    for dy in range(-r, r+1):
        for dx in range(-r, r+1):
            bb = np.roll(np.roll(b, dy, 0), dx, 1)[r:-r, r:-r].astype(np.float32)
            v = float(np.sqrt(np.mean((ac - bb)**2)) / 255.)
            if v < best[0]: best = (v, ac, bb)
    return best

NR = np.zeros(len(ci), np.float32); SS = np.zeros(len(ci), np.float32)
t0 = time.perf_counter()
for k in tqdm(range(len(ci)), desc="aligned nRMSE+SSIM", leave=False):
    v, aa, bb = aligned(U[ci[k]], U[cj[k]])
    NR[k] = v; SS[k] = ssim(aa, bb, data_range=255)             # aligned SSIM in the metric loop
print(f"metrics {time.perf_counter()-t0:.0f}s", flush=True)

is_md5 = np.array([MD5[a] is not None and MD5[a] == MD5[b] for a, b in zip(ci, cj)])
near = (NR <= NR_TH) & (SS >= SS_TH)
print(f"\n--- calibration on {int(is_md5.sum())} MD5 positives ---", flush=True)
if is_md5.sum():
    print(f"  nRMSE max={NR[is_md5].max():.4f}   SSIM min={SS[is_md5].min():.4f}", flush=True)
print(f"verified near-dups: {int(near.sum())} (md5 {int((near&is_md5).sum())}, new {int((near&~is_md5).sum())})", flush=True)

par = list(range(N)); mem = {i: [i] for i in range(N)}
def find(x):
    while par[x] != x: par[x] = par[par[x]]; x = par[x]
    return x

def ok(a, b):
    ra, rb = find(a), find(b)
    if ra == rb: return False
    A_, B_ = mem[ra], mem[rb]
    if len(A_) + len(B_) > CAP: return False
    for p in A_:
        for q in B_:
            if MD5[p] is not None and MD5[p] == MD5[q]: continue
            v, aa, bb = aligned(U[p], U[q])
            if v > NR_TH or ssim(aa, bb, data_range=255) < SS_TH: return False   # FIX: aligned SSIM here too
    par[rb] = ra; mem[ra] = A_ + B_; mem.pop(rb); return True

order = np.argsort(NR); nm = 0
for k in order:
    if not (is_md5[k] or near[k]): continue
    if ok(int(ci[k]), int(cj[k])): nm += 1
print(f"merges applied: {nm}", flush=True)

root = np.array([find(i) for i in range(N)]); _, gid = np.unique(root, return_inverse=True)
df["_group"] = "g" + pd.Series(gid).astype(str)
vc = df._group.value_counts()
print("--- group size histogram (final) ---", flush=True)
print(vc.value_counts().sort_index().to_string(), flush=True)
print(f"groups={df._group.nunique()} max={vc.max()} at_cap={int((vc==CAP).sum())}", flush=True)
nun = df.groupby("_group").diagnosis.nunique(); bad = nun[nun > 1].index
multi = int((vc > 1).sum())
print(f"multi={multi} conflicts={len(bad)} ({100*len(bad)/max(1,multi):.1f}%) dropped={int(df._group.isin(bad).sum())}", flush=True)
clean = df[~df._group.isin(bad)].reset_index(drop=True)
print(f"FINAL POOL: {len(clean)} images / {clean._group.nunique()} groups", flush=True)
print("grades:", clean.diagnosis.value_counts().sort_index().to_dict(), flush=True)

pd.DataFrame(dict(i=ci, j=cj, nrmse=NR, ssim=SS, md5=is_md5, verified=near|is_md5)).to_csv(OUT/"dup_pairs_final.csv", index=False)
df[["id_code","_group"]].to_csv(OUT/"groups_final_all.csv", index=False)
clean[["id_code","_group"]].to_csv(OUT/"groups_final.csv", index=False)
(OUT/"dedup_final_summary.json").write_text(json.dumps(dict(
    S=S, R=R, HAM=HAM, NR_TH=NR_TH, SS_TH=SS_TH, CAP=CAP,
    n_raw=N, candidates=int(len(ci)), md5_pos=int(is_md5.sum()), verified=int(near.sum()), merges=nm,
    groups=int(df._group.nunique()), max_group=int(vc.max()), at_cap=int((vc==CAP).sum()),
    multi=multi, conflicts=int(len(bad)), dropped=int(df._group.isin(bad).sum()),
    clean_pool=len(clean), clean_groups=int(clean._group.nunique())), indent=2))

rng = np.random.default_rng(0); rows = []
for sel, lab in [(np.where(near & ~is_md5)[0], "VERIFIED"), (np.where((NR<=0.06) & (SS>=0.85) & ~near)[0], "BORDER-REJ")]:
    if len(sel) == 0: continue
    for k in rng.choice(sel, min(8, len(sel)), replace=False):
        a, b = int(ci[k]), int(cj[k])
        ia = cv2.resize(cv2.imread(paths[a]), (224,224)); ib = cv2.resize(cv2.imread(paths[b]), (224,224))
        cv2.putText(ia, f"nr{NR[k]:.3f}", (4,18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
        cv2.putText(ib, f"ss{SS[k]:.3f}", (4,18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 2)
        cv2.putText(ia, f"g{grades[a]}", (4,214), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)
        cv2.putText(ib, f"g{grades[b]} {lab}", (4,214), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,0), 2)
        rows.append(np.hstack([ia, ib]))
if rows: cv2.imwrite(str(OUT/"dup_audit_final.jpg"), cv2.vconcat(rows)); print("audit ->", OUT/"dup_audit_final.jpg", flush=True)

# ---------- folds_final.json ----------
def gh(d, n, seed):
    try:
        s = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed)
        r, p = next(s.split(d, d.diagnosis, groups=d._group)); return d.iloc[r].copy(), d.iloc[p].copy()
    except Exception:
        s = GroupShuffleSplit(n_splits=1, test_size=1./n, random_state=seed)
        r, p = next(s.split(d, groups=d._group)); return d.iloc[r].copy(), d.iloc[p].copy()

FJ = OUT/"folds_final.json"
if not FJ.exists():
    folds = {}
    for f, (di, ti) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(clean, clean.diagnosis, groups=clean._group)):
        dev, test = clean.iloc[di].copy(), clean.iloc[ti].copy()
        rem, tune = gh(dev, 8, SEED+f); rem, ca = gh(rem, 7, SEED+f+100); fit, cb = gh(rem, 6, SEED+f+200)
        parts = dict(train=fit, tune=tune, cal_a=ca, cal_b=cb, test=test)
        ids = {k: p.id_code.tolist() for k, p in parts.items()}; gs = {k: set(p._group) for k, p in parts.items()}
        for a_ in ids:
            for b_ in ids:
                if a_ < b_:
                    assert not(set(ids[a_]) & set(ids[b_])), (a_, b_, "img leak")
                    assert not(gs[a_] & gs[b_]), (a_, b_, "GROUP LEAK")
        folds[str(f)] = ids
    FJ.write_text(json.dumps(dict(seed=SEED, n=len(clean), folds=folds), indent=2))
    print("saved folds_final.json", flush=True)

meta = json.loads(FJ.read_text())
byid = clean.set_index("id_code")
P0 = {k: byid.loc[meta["folds"]["0"][k]] for k in ["train",tune","cal_a","cal_b","test"]}
print("fold0 sizes:", {k: len(v) for k, v in P0.items()}, flush=True)
for k in ["train","test"]:
    vcg = P0[k].diagnosis.value_counts().reindex(range(5), fill_value=0)
    print(f"  {k}: " + ", ".join(f"g{g}={int(vcg[g])}" for g in range(5)), flush=True)
print("\nREADY: GRP=groups_final.csv, FJ=folds_final.json. Launch Fold 0.", flush=True)

APTOS raw: 3662 | grades [1805, 370, 999, 193, 295]


md5+FOV thumb:   0%|          | 0/3662 [00:00<?, ?it/s]

candidates (Hamming<=10): 61596


aligned nRMSE+SSIM:   0%|          | 0/61596 [00:00<?, ?it/s]

metrics 313s

--- calibration on 134 MD5 positives ---
  nRMSE max=0.0000   SSIM min=1.0000
verified near-dups: 162 (md5 134, new 28)
merges applied: 148
--- group size histogram (final) ---
count
1    3376
2     129
3       8
4       1
groups=3514 max=4 at_cap=0
multi=138 conflicts=37 (26.8%) dropped=79
FINAL POOL: 3583 images / 3477 groups
grades: {0: 1801, 1: 360, 2: 975, 3: 172, 4: 275}
audit -> /kaggle/working/cgrl/dup_audit_final.jpg
saved folds_final.json
fold0 sizes: {'train': 1788, 'tune': 354, 'cal_a': 361, 'cal_b': 361, 'test': 719}
  train: g0=889, g1=167, g2=504, g3=87, g4=141
  test: g0=368, g1=75, g2=193, g3=28, g4=55

READY: GRP=groups_final.csv, FJ=folds_final.json. Launch Fold 0.


In [ ]:
# ================= VM REPAIR =================
import sys, types, importlib
import torch
class _FakeMod(types.ModuleType):
    def __getattr__(s,n):
        if n.startswith("__"): raise AttributeError(n)
        def _noop(*a,**k): return a[0] if a else None
        return _noop
try:
    importlib.import_module("torch._utils"); import torch._dynamo; print("dynamo: REAL",flush=True)
except Exception as e:
    print("dynamo shim:",type(e).__name__,flush=True)
    fd=_FakeMod("torch._dynamo"); fu=_FakeMod("torch._dynamo.utils"); fu.is_compile_supported=lambda:False
    fe=_FakeMod("torch._dynamo.eval_frame"); fc=_FakeMod("torch._dynamo.config")
    fd.utils,fd.eval_frame,fd.config=fu,fe,fc; fd.is_compiling=lambda:False
    sys.modules.update({"torch._dynamo":fd,"torch._dynamo.utils":fu,"torch._dynamo.eval_frame":fe,"torch._dynamo.config":fc})
    torch._dynamo=fd; torch.compiler.disable=lambda fn=None,recursive=True,reason=None:(fn if callable(fn) else (lambda f:f))
for _m in ("torchvision","timm"):
    try: importlib.import_module(_m); print("import OK:",_m,flush=True)
    except Exception as e: print("import FAIL:",_m,e,flush=True)
import timm
from tqdm.auto import tqdm
sys.unraisablehook=lambda *a,**k: None

import os,json,math,time,copy,random,platform
from pathlib import Path
import numpy as np,pandas as pd,cv2,torch.nn as nn,torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold,GroupShuffleSplit,train_test_split
from sklearn.metrics import cohen_kappa_score,accuracy_score,f1_score,confusion_matrix,roc_auc_score

# ================= CONFIG (FOLD 0, GROUPED ARM) =================
MODE="pilot"; VARIANT="full"; SEED=2026; FOLD=0; GROUP_AWARE=True
BASE="/kaggle/input/datasets/akashbisht77273/dataset231243"
APTOS=Path("/kaggle/input/competitions/aptos2019-blindness-detection")
FG=Path(BASE)/"FGADR-Seg-set_Release/FGADR-Seg-set_Release/Seg-set"
FGCSV=FG/"DR_Seg_Grading_Label.csv"
IDR=Path(BASE)/"A. Segmentation/A. Segmentation"
M2_DIR=Path("/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2")
M2_CSV=Path("/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv")
DDR_DIR=Path("/kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading/DR_grading")
DDR_CSV=Path("/kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading.csv")
OUT=Path("/kaggle/working/cgrl"); OUT.mkdir(parents=True,exist_ok=True)
TAG="grouped" if GROUP_AWARE else "random"
RUN=OUT/f"{VARIANT}_f{FOLD}_{TAG}_final"; RUN.mkdir(parents=True,exist_ok=True)
FIGS=RUN/"epoch_figs"; FIGS.mkdir(exist_ok=True)

CONCEPTS=["MA","HE","EX","SE","VB","IRMA","NV","VH"]; CI={c:i for i,c in enumerate(CONCEPTS)}
FG_C=["MA","HE","EX","SE","IRMA","NV"]; ID_C=["MA","HE","EX","SE"]
ABs=dict(full=dict(ctx=1,zoom=1,graph=1,nodes=1,ground=1,strict=0),
         strict=dict(ctx=1,zoom=1,graph=1,nodes=1,ground=1,strict=1),
         graph_free=dict(ctx=1,zoom=1,graph=0,nodes=1,ground=1,strict=0),
         baseline=dict(ctx=0,zoom=0,graph=0,nodes=0,ground=0,strict=0),
         no_vb_irma=dict(ctx=1,zoom=1,graph=1,nodes=1,ground=1,strict=0))
def set_variant(V):
    global AB,KEEP,NCLIN,KN,AUX_ON
    AB=ABs[V]; KEEP=[0,1,2,3,6,7] if V=="no_vb_irma" else list(range(8))
    NCLIN=len(KEEP); KN=[CONCEPTS[i] for i in KEEP]; AUX_ON=AB["nodes"]==1
set_variant(VARIANT)
C=dict(hr=1024,lr=384,patch=160,regions=3,grid=8,mask=128,
       batch=4,accum=4,eval_batch=2,aux_batch=4,
       epochs=22,stage1=13,tau=3.0,tta=4,
       fusion=512,cdim=256,latent=2,graph_layers=2,
       head_lr=3e-4,lr_ratio=0.1,wd=0.01,dropout=0.1,mixup_alpha=0.3,
       gamma=0.5,clip=1.0,warmup=0.1,stage2_mult=0.25,cos_floor=0.01,
       max_gib=13.5,boot=2000,thr_boot=50,workers=4,gate_init=1.0,
       aux_w=0.75,aux_every=2,epoch_figs=4,cam_images=6)
MASKSZ=C["mask"]; GI=C["gate_init"]
DEV="cuda" if torch.cuda.is_available() else "cpu"; cv2.setNumThreads(1)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True
def amp_scaler(on):
    try: return torch.amp.GradScaler("cuda",enabled=on)
    except TypeError: return torch.cuda.amp.GradScaler(enabled=on)
def amp_ctx(on):
    try: return torch.amp.autocast("cuda",dtype=torch.float16,enabled=on)
    except TypeError: return torch.cuda.amp.autocast(enabled=on,dtype=torch.float16)
def banner(t): print("\n"+"="*98+f"\n{t}\n"+"="*98,flush=True)
_trapz=getattr(np,"trapezoid",getattr(np,"trapz",None))
def dist_report(name,d,col="diagnosis"):
    vc=d[col].value_counts().reindex(range(5),fill_value=0); tot=int(vc.sum())
    print(f"{name:12s} n={tot:5d} | "+"  ".join(f"g{g}:{int(vc[g]):4d}({100*vc[g]/max(1,tot):5.1f}%)" for g in range(5)),flush=True)
def qwk_fast(y,p,K=5):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    O=np.bincount(y*K+p,minlength=K*K).reshape(K,K).astype(np.float64)
    W=((np.arange(K)[:,None]-np.arange(K)[None,:])**2)/(K-1)**2
    E=np.outer(O.sum(1),O.sum(0))/max(O.sum(),1); d=(W*E).sum()
    return float(1-(W*O).sum()/d) if d>0 else 0.0
def full_metrics(y,pred,prob=None):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    cm=confusion_matrix(y,pred,labels=list(range(5))); tot=int(cm.sum()); per={}
    for g in range(5):
        tp=int(cm[g,g]); fn=int(cm[g,:].sum()-tp); fp=int(cm[:,g].sum()-tp); tn=tot-tp-fn-fp
        per[g]=dict(n=int(cm[g,:].sum()),pred=int(cm[:,g].sum()),
                    sens=(tp/(tp+fn)) if tp+fn>0 else None,
                    spec=(tn/(tn+fp)) if tn+fp>0 else None,
                    prec=(tp/(tp+fp)) if tp+fp>0 else None,
                    f1=(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn>0 else 0.0)
    recs=[per[g]["sens"] for g in range(5) if per[g]["sens"] is not None]
    d=dict(acc=float(accuracy_score(y,pred)),qwk=float(cohen_kappa_score(y,pred,weights="quadratic")),
           qwk_lin=float(cohen_kappa_score(y,pred,weights="linear")),
           f1m=float(f1_score(y,pred,average="macro",zero_division=0)),
           f1w=float(f1_score(y,pred,average="weighted",zero_division=0)),
           bal_acc=float(np.mean(recs)) if recs else 0.0,cm=cm.tolist(),per_class=per)
    if prob is not None:
        try: d["auc"]=float(roc_auc_score(y,prob,multi_class="ovr",average="macro"))
        except Exception: d["auc"]=None
        try: d["ref_auc"]=float(roc_auc_score((y>=2).astype(int),prob@np.arange(5))) if len(np.unique(y>=2))>1 else None
        except Exception: d["ref_auc"]=None
        conf=prob.max(1); ok=(prob.argmax(1)==y).astype(float); b=np.linspace(0,1,16); e=0.
        for i in range(15):
            m=(conf>b[i])&(conf<=b[i+1])
            if m.sum(): e+=m.mean()*abs(ok[m].mean()-conf[m].mean())
        d["ece"]=float(e)
        oh=np.zeros_like(prob); oh[np.arange(len(y)),y]=1
        d["brier"]=float(((prob-oh)**2).sum(1).mean())
        d["nll"]=float(-np.log(prob[np.arange(len(y)),y].clip(1e-9)).mean())
    return d
def print_block(tag,M,extra=""):
    print(f"{tag} | acc {M['acc']:.4f} bal_acc {M['bal_acc']:.4f} qwk {M['qwk']:.4f} qwk_lin {M['qwk_lin']:.4f} "
          f"f1m {M['f1m']:.4f} f1w {M['f1w']:.4f}"+(f" auc {M['auc']:.4f} ref_auc {M['ref_auc']:.4f}" if M.get('auc') is not None else "")+extra,flush=True)
    print("   cls    n  pred    sens    spec    prec      f1",flush=True)
    for g,d in M["per_class"].items():
        f=lambda v:"  ---  " if v is None else f"{v:7.3f}"
        print(f"    {g}  {d['n']:4d} {d['pred']:5d} {f(d['sens'])} {f(d['spec'])} {f(d['prec'])} {d['f1']:7.3f}",flush=True)
    print("   confusion (rows=true, cols=pred):",flush=True)
    for r in M["cm"]: print("     "+" ".join(f"{v:5d}" for v in r),flush=True)

banner(f"FOLD {FOLD} | ARM {TAG} | VARIANT {VARIANT} | hr{C['hr']} lr{C['lr']} patch{C['patch']} grid{C['grid']} mask{MASKSZ} gate_init{GI}")

# ---------- POOL + SPLITS (final dedup files) ----------
GRP=OUT/"groups_final.csv"
g=pd.read_csv(GRP)
df_raw=pd.read_csv(APTOS/"train.csv"); df_raw["path"]=df_raw.id_code.map(lambda x:str(APTOS/"train_images"/f"{x}.png"))
POOL=df_raw.merge(g,on="id_code",how="inner").reset_index(drop=True)
assert set(POOL.id_code)==set(g.id_code)
(OUT/f"pool_ids_{TAG}.txt").write_text("\n".join(sorted(POOL.id_code)))
print(f"[{TAG}] clean pool: {len(POOL)} images / {POOL._group.nunique()} groups",flush=True)
dist_report("POOL",POOL)
def gh(d,n,seed):
    try:
        s=StratifiedGroupKFold(n_splits=n,shuffle=True,random_state=seed)
        r,p=next(s.split(d,d.diagnosis,groups=d._group)); return d.iloc[r].copy(),d.iloc[p].copy()
    except Exception:
        s=GroupShuffleSplit(n_splits=1,test_size=1./n,random_state=seed)
        r,p=next(s.split(d,groups=d._group)); return d.iloc[r].copy(),d.iloc[p].copy()
if GROUP_AWARE:
    FJ=OUT/"folds_final.json"; meta=json.loads(FJ.read_text()); src=meta["folds"][str(FOLD)]
else:
    FJ=OUT/f"folds_random_{SEED}.json"
    if FJ.exists(): meta=json.loads(FJ.read_text())
    else:
        dev,test=train_test_split(POOL,test_size=.20,random_state=SEED,stratify=POOL.diagnosis)
        tr,rest=train_test_split(dev,test_size=.375,random_state=SEED,stratify=dev.diagnosis)
        tune,cal=train_test_split(rest,test_size=1/3,random_state=SEED,stratify=rest.diagnosis)
        ca,cb=train_test_split(cal,test_size=.5,random_state=SEED,stratify=cal.diagnosis)
        meta=dict(seed=SEED,tag="random_nogroup",n=len(POOL),
                  ids={k:p.id_code.tolist() for k,p in dict(train=tr,tune=tune,cal_a=ca,cal_b=cb,test=test).items()})
        FJ.write_text(json.dumps(meta,indent=2))
    src=meta["ids"]
byid=POOL.set_index("id_code")
P={k:byid.loc[src[k]].reset_index(drop=True) for k in ["train","tune","cal_a","cal_b","test"]}
assert set().union(*[set(v) for v in src.values()])==set(POOL.id_code)
if GROUP_AWARE:
    gs={k:set(P[k]._group) for k in P}
    for a in gs:
        for b in gs:
            if a<b: assert not(gs[a]&gs[b]),f"GROUP LEAK {a}/{b}"
    print("[CHECK] group-confined: 0 duplicate-groups cross any split",flush=True)
else:
    ov=set(P["train"]._group)&set(P["test"]._group)
    print(f"[DISCLOSE] random arm: {len(ov)} duplicate-groups shared train/test",flush=True)
print("--- CLASS DISTRIBUTION ---",flush=True)
for k in ["train","tune","cal_a","cal_b","test"]: dist_report(k.upper(),P[k])

# ---------- AUX MANIFEST (BUILD IF MISSING, ELSE CACHE) ----------
FGF={"MA":"Microaneurysms_Masks","HE":"Hemohedge_Masks","EX":"HardExudate_Masks",
     "SE":"SoftExudate_Masks","NV":"Neovascularization_Masks","IRMA":"IRMA_Masks"}
IDMAP={1:"MA",2:"HE",3:"EX",4:"SE",5:"OD"}
AJ=OUT/"aux_manifest.json"
if AJ.exists():
    aux=json.loads(AJ.read_text()); print("cached aux:",len(aux),flush=True)
else:
    def ls(f):
        f=Path(f); return {} if not f.exists() else {Path(n).stem:n for n in os.listdir(f)}
    raw=pd.read_csv(FGCSV,header=None)
    hdr=not str(raw.iloc[0,0]).lower().endswith((".png",".jpg",".jpeg"))
    fg=pd.read_csv(FGCSV) if hdr else raw
    idc=fg.columns[0]; gc=fg.columns[1] if fg.shape[1]>1 else None
    if gc is not None and not pd.api.types.is_numeric_dtype(fg[gc]): gc=None
    im=ls(FG/"Original_Images"); mm={c:ls(FG/f) for c,f in FGF.items()}
    _,ho=train_test_split(fg,test_size=.2,stratify=fg[gc],random_state=SEED) if gc is not None else train_test_split(fg,test_size=.2,random_state=SEED)
    hoids=set(ho[idc].astype(str)); aux=[]
    for _,r in fg.iterrows():
        st=Path(str(r[idc]).strip()).stem
        if st not in im: continue
        mk={c:str(FG/FGF[c]/m[st]) for c,m in mm.items() if st in m}
        aux.append(dict(kind="fgadr",image=str(FG/"Original_Images"/im[st]),masks=mk,gt=None,
                        holdout=(str(r[idc]) in hoids),grade=(int(r[gc]) if gc is not None else -1)))
    for sp,(idir,gdir) in {"train":(IDR/"1. Original Images"/"a. Training Set",IDR/"2. All Segmentation Groundtruths"/"a. Training Set"),
                           "test":(IDR/"1. Original Images"/"b. Testing Set",IDR/"2. All Segmentation Groundtruths"/"b. Testing Set")}.items():
        if not idir.exists() or not gdir.exists(): continue
        ims=ls(idir); gts=[f for f in gdir.rglob("*") if f.is_file()]
        lab=any(Path(f).stem in ims for f in gts[:20])
        for st,nm in ims.items():
            if lab:
                m=[f for f in gts if Path(f).stem==st]
                if m: aux.append(dict(kind="idrid",image=str(idir/nm),masks={},gt=str(m[0]),holdout=(sp=="test"),grade=-1))
            else:
                mk={t:str(next(f for f in gts if st in Path(f).stem and t in Path(f).stem.upper()))
                    for t in ID_C if any(st in Path(f).stem and t in Path(f).stem.upper() for f in gts)}
                aux.append(dict(kind="idrid",image=str(idir/nm),masks=mk,gt=None,holdout=(sp=="test"),grade=-1))
    AJ.write_text(json.dumps(aux)); print("built aux:",len(aux),flush=True)
assert len(aux)>0, "aux manifest empty"
aux_tr=[r for r in aux if not r["holdout"]]; aux_ho=[r for r in aux if r["holdout"]]
print(f"aux train={len(aux_tr)} holdout={len(aux_ho)}",flush=True)

# ---------- DATA ----------
def geom(b):
    h,w=b.shape[:2]; sc=256./max(h,w)
    sm=cv2.resize(b,(max(1,int(w*sc)),max(1,int(h*sc))),interpolation=cv2.INTER_AREA)
    gg=cv2.cvtColor(sm,cv2.COLOR_BGR2GRAY); _,th=cv2.threshold(gg,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    cs,_=cv2.findContours(th,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not cs: return 0,0,w,h
    x,y,ww,hh=cv2.boundingRect(max(cs,key=cv2.contourArea)); x,y,ww,hh=[int(v/sc) for v in (x,y,ww,hh)]
    return max(0,x),max(0,y),min(ww,w-x),min(hh,h-y)
def sqcrop(a,x,y,ww,hh):
    s=max(ww,hh); o=np.zeros((s,s)+a.shape[2:],a.dtype); px,py=(s-ww)//2,(s-hh)//2
    o[py:py+hh,px:px+ww]=a[y:y+hh,x:x+ww]; return o
def mdown(m,out):
    h,w=m.shape[:2]
    if h<out or w<out:
        m=cv2.resize(m,(max(out,w),max(out,h)),interpolation=cv2.INTER_NEAREST); h,w=m.shape[:2]
    bh,bw=h//out,w//out
    return (m[:bh*out,:bw*out].reshape(out,bh,out,bw).max(axis=(1,3))>0).astype(np.uint8)
def clahe_rgb(rgb):
    lab=cv2.cvtColor(rgb,cv2.COLOR_RGB2LAB); lab[...,0]=cv2.createCLAHE(2.,(8,8)).apply(lab[...,0])
    return cv2.cvtColor(lab,cv2.COLOR_LAB2RGB)
def ends(b):
    x,y,ww,hh=geom(b); s=sqcrop(b,x,y,ww,hh)
    rgb=cv2.cvtColor(cv2.resize(s,(C["hr"],C["hr"]),interpolation=cv2.INTER_AREA),cv2.COLOR_BGR2RGB)
    return clahe_rgb(rgb),rgb,(x,y,ww,hh)
def augment(cl,gr,mk=None):
    if random.random()<.5:
        cl,gr=cl[:,::-1],gr[:,::-1]
        if mk is not None: mk=mk[:,:,::-1]
    if random.random()<.5:
        cl,gr=cl[::-1],gr[::-1]
        if mk is not None: mk=mk[:,::-1]
    k=random.randrange(4)
    if k:
        cl,gr=np.rot90(cl,k),np.rot90(gr,k)
        if mk is not None: mk=np.rot90(mk,k,axes=(1,2))
    if random.random()<.5:
        h,w=cl.shape[:2]; M=cv2.getRotationMatrix2D((w/2,h/2),random.uniform(-15,15),random.uniform(.9,1.1))
        M[0,2]+=random.uniform(-.05,.05)*w; M[1,2]+=random.uniform(-.05,.05)*h
        cl=cv2.warpAffine(cl,M,(w,h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT)
        gr=cv2.warpAffine(gr,M,(w,h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT)
        if mk is not None:
            mh,mw=mk.shape[1:]
            mk=np.stack([cv2.warpAffine(mk[i],M,(mw,mh),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT,borderValue=0) for i in range(mk.shape[0])])
    if random.random()<.5:
        a,b_=random.uniform(.9,1.1),random.uniform(-15,15)
        cl=np.clip(cl.astype(np.float32)*a+b_,0,255).astype(np.uint8)
        gr=np.clip(gr.astype(np.float32)*a+b_,0,255).astype(np.uint8)
    return np.ascontiguousarray(cl),np.ascontiguousarray(gr),(None if mk is None else np.ascontiguousarray(mk))
class PDS(Dataset):
    def __init__(s,d,train): s.d=d.reset_index(drop=True); s.train=train
    def __len__(s): return len(s.d)
    def __getitem__(s,i):
        r=s.d.iloc[i]; cl,raw,_=ends(cv2.imread(r.path))
        if s.train: cl,raw,_=augment(cl,raw)
        hr=np.stack([cl,raw]).transpose(0,3,1,2).copy()
        t=np.zeros(5,np.float32); t[int(r.diagnosis)]=1
        return dict(hr=torch.from_numpy(hr).float()/255,target=torch.from_numpy(t),grade=torch.tensor(int(r.diagnosis)))
class ADS(Dataset):
    def __init__(s,rows,train=True): s.rows=rows; s.train=train
    def __len__(s): return len(s.rows)
    def __getitem__(s,i):
        r=s.rows[i]; bgr=cv2.imread(r["image"])
        cl,raw,(x,y,ww,hh)=ends(bgr)
        mk=np.zeros((8,MASKSZ,MASKSZ),np.uint8); kn=np.zeros(8,np.uint8)
        for c in (FG_C if r["kind"]=="fgadr" else ID_C): kn[CI[c]]=1
        if r.get("gt"):
            gg=cv2.imread(r["gt"],cv2.IMREAD_UNCHANGED)
            if gg is not None and gg.ndim==2:
                gg=sqcrop(gg,x,y,ww,hh)
                for v,c in IDMAP.items():
                    if c=="OD": continue
                    mk[CI[c]]=mdown((gg==v).astype(np.uint8),MASKSZ)
        else:
            for c,mp in r["masks"].items():
                m=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
                if m is None: continue
                mk[CI[c]]=mdown(sqcrop(m,x,y,ww,hh),MASKSZ)
        pres=np.full(8,-1.,np.float32)
        for j in range(8):
            if kn[j]: pres[j]=float(mk[j].any())
        if s.train: cl,raw,mk=augment(cl,raw,mk)
        hr=np.stack([cl,raw]).transpose(0,3,1,2).copy()
        return dict(hr=torch.from_numpy(hr).float()/255,masks=torch.from_numpy(mk.astype(np.float32)),
                    mask_valid=torch.from_numpy(kn.astype(np.float32)),concept=torch.from_numpy(pres),
                    grade=torch.tensor(int(r["grade"])))
def ploader(d,train,bal,ep):
    gen=torch.Generator().manual_seed(SEED+ep); smp=None
    if bal:
        cnt=np.bincount(d.diagnosis.values,minlength=5).clip(1)
        w=[cnt[y]**(-(1+C["gamma"])) for y in d.diagnosis.values]
        smp=WeightedRandomSampler(torch.tensor(w),len(d),replacement=True,generator=gen)
    return DataLoader(PDS(d,train),batch_size=C["batch"] if train else C["eval_batch"],
                      shuffle=train and smp is None,sampler=smp,num_workers=C["workers"],pin_memory=True,generator=gen)
def mixup(b,al):
    y=b["grade"]; n=len(y)
    if n<2 or al<=0: return b
    pm=torch.randperm(n,device=y.device)
    lam=torch.distributions.Beta(al,al).sample((n,)).to(y.device)
    lam=torch.where((y-y[pm]).abs()<=1,lam,torch.ones_like(lam))
    o=dict(b); w=lam[:,None,None,None,None]
    o["hr"]=w*b["hr"]+(1-w)*b["hr"][pm]; o["target"]=lam[:,None]*b["target"]+(1-lam[:,None])*b["target"][pm]
    return o

# ---------- MODEL ----------
def cv_(ci,co,k=3,s=1,g=1): return nn.Sequential(nn.Conv2d(ci,co,k,s,k//2,groups=g),nn.GroupNorm(1,co),nn.GELU())
def rs(x,hw): return F.interpolate(x,size=hw,mode="bilinear",align_corners=False)
def graham_gpu(x):
    xs=F.avg_pool2d(x,4)
    sig=max(1.0,(C["hr"]/4)/30.); k=int(2*round(3*sig)+1)
    c=torch.arange(k,device=x.device,dtype=x.dtype)-k//2
    gg=torch.exp(-c**2/(2*sig**2)); gg=(gg/gg.sum()).view(1,1,1,k)
    y=F.conv2d(F.pad(xs,(k//2,k//2,0,0),mode="reflect"),gg.expand(3,1,1,k),groups=3)
    y=F.conv2d(F.pad(y,(0,0,k//2,k//2),mode="reflect"),gg.transpose(2,3).expand(3,1,k,1),groups=3)
    y=F.interpolate(y,size=x.shape[-2:],mode="bilinear",align_corners=False)
    return (4*x-4*y+128/255.).clamp(0,1)
class CA(nn.Module):
    def __init__(s,d):
        super().__init__(); m=max(16,d//16); s.r,s.h,s.w=cv_(d,m,1),nn.Conv2d(m,d,1),nn.Conv2d(m,d,1)
    def forward(s,x):
        h,w=x.shape[-2:]
        z=s.r(torch.cat([x.mean(3,keepdim=True),x.mean(2,keepdim=True).transpose(2,3)],2))
        zh,zw=z.split([h,w],2); return x*s.h(zh).sigmoid()*s.w(zw.transpose(2,3)).sigmoid()
class Fusion(nn.Module):
    def __init__(s,ch):
        super().__init__(); d=C["fusion"]; s.p=nn.ModuleList([cv_(c,d,1) for c in ch])
        s.hm,s.ml,s.wt=nn.Conv2d(2*d,d,1),nn.Conv2d(2*d,d,1),nn.Conv2d(3*d,3,1); s.ca=CA(d)
    def forward(s,fs):
        lo,mid,hi=[p(x) for p,x in zip(s.p,fs)]
        up=rs(hi,mid.shape[-2:]); mid=mid+s.hm(torch.cat([mid,up],1)).sigmoid()*up
        up=rs(mid,lo.shape[-2:]); lo=lo+s.ml(torch.cat([lo,up],1)).sigmoid()*up
        mp=[lo,rs(mid,lo.shape[-2:]),rs(hi,lo.shape[-2:])]
        w=s.wt(torch.cat(mp,1)).softmax(1); return s.ca(sum(w[:,i:i+1]*mp[i] for i in range(3)))
class Ctx(nn.Module):
    def __init__(s,st,kv=16):
        super().__init__(); d=C["fusion"]; s.st,s.kv=st,kv
        s.q,s.k,s.v,s.o=nn.Conv2d(d,128,1),nn.Conv2d(d,128,1),nn.Conv2d(d,d,1),nn.Conv2d(d,d,1)
        s.g=nn.Parameter(torch.full((),GI))
    def forward(s,x):
        sm=F.avg_pool2d(x,s.st); kv=F.adaptive_avg_pool2d(x,s.kv); b,_,h,w=sm.shape
        q=s.q(sm).flatten(2).transpose(1,2)[:,None]; k=s.k(kv).flatten(2).transpose(1,2)[:,None]; v=s.v(kv).flatten(2).transpose(1,2)[:,None]
        z=F.scaled_dot_product_attention(q,k,v)[:,0].transpose(1,2).reshape(b,-1,h,w)
        return x+s.g*rs(s.o(z),x.shape[-2:])
class Zoom(nn.Module):
    def __init__(s):
        super().__init__(); d,p,R,G=C["fusion"],C["patch"],C["regions"],C["grid"]; s.R,s.p,s.G=R,p,G
        s.sal,s.loc=nn.Conv2d(d,1,1),nn.Conv2d(d,R*2,1)
        s.enc=nn.Sequential(cv_(3,32,3,2),cv_(32,32,3,2,32),cv_(32,64,1),cv_(64,64,3,2,64),
                            cv_(64,128,1),cv_(128,128,3,2,128),cv_(128,d,1))
        s.gate=nn.Parameter(torch.full((),GI))
    def forward(s,x,hr):
        B,_,H,W=hr.shape; G=s.G
        sal=s.sal(F.adaptive_avg_pool2d(x,G)).flatten(1)
        idx=sal.topk(s.R,dim=1).indices
        cy=((idx//G).float()+.5)/G*2-1; cx=((idx%G).float()+.5)/G*2-1
        base=torch.stack([cx,cy],-1); sc=s.p/float(H)
        res=torch.tanh(s.loc(F.adaptive_avg_pool2d(x,1)).flatten(1)).reshape(B,s.R,2)
        off=(base+.5*res).clamp(-1,1)*(1-sc)
        rep=hr[:,None].expand(-1,s.R,-1,-1,-1).reshape(B*s.R,3,H,W)
        th=torch.zeros(B*s.R,2,3,device=x.device,dtype=x.dtype)
        th[:,0,0]=sc; th[:,1,1]=sc; th[:,:,2]=off.reshape(B*s.R,2)
        grid=F.affine_grid(th,(B*s.R,3,s.p,s.p),align_corners=False)
        pat=F.grid_sample(rep,grid,mode="bilinear",padding_mode="zeros",align_corners=False)
        z=s.enc(pat).reshape(B,s.R,C["fusion"],s.p//16,s.p//16)
        w=torch.softmax(sal.gather(1,idx),dim=1)
        z=(z*w[...,None,None,None]).sum(1)
        return x+s.gate*rs(z,x.shape[-2:]),sal
class Ground(nn.Module):
    def __init__(s,d,fd):
        super().__init__(); s.q=nn.Linear(d,d); s.k=nn.Conv2d(fd,d,1); s.v=nn.Conv2d(fd,d,1)
        s.ln=nn.LayerNorm(d); s.sc=d**-.5
    def forward(s,n,f):
        B,N,D=n.shape; H,W=f.shape[-2:]
        a=torch.softmax(s.q(n)@s.k(f).flatten(2)*s.sc,-1)
        return s.ln(n+a@s.v(f).flatten(2).transpose(1,2)),a.reshape(B,N,H,W)
class Graph(nn.Module):
    def __init__(s,d): super().__init__(); s.a=nn.MultiheadAttention(d,1,batch_first=True); s.g=nn.GRUCell(d,d)
    def forward(s,x):
        o,w=s.a(x,x,x,need_weights=True); b,n,d=x.shape
        return s.g(o.reshape(-1,d),x.reshape(-1,d)).reshape(b,n,d),w
class Net(nn.Module):
    def __init__(s):
        super().__init__()
        try: s.bb=timm.create_model("convnextv2_tiny.fcmae_ft_in22k_in1k",pretrained=True,features_only=True,out_indices=(0,1,2),drop_path_rate=.05)
        except Exception as e:
            print("random init:",e,flush=True); s.bb=timm.create_model("convnextv2_tiny",pretrained=False,features_only=True,out_indices=(0,1,2))
        ch=s.bb.feature_info.channels()
        s.fusion=Fusion(ch); s.ctx=nn.ModuleList([Ctx(2),Ctx(4)]); s.zoom=Zoom(); s.alpha=nn.Parameter(torch.zeros(()))
        s.summ=nn.Sequential(nn.Linear(2*C["fusion"],C["fusion"]),nn.GELU(),nn.Dropout(C["dropout"]))
        s.dyn=nn.Linear(C["fusion"],NCLIN*C["cdim"]); s.lat=nn.Linear(C["fusion"],C["latent"]*C["cdim"])
        s.ground=Ground(C["cdim"],C["fusion"]); s.pres=nn.Linear(C["cdim"],1)
        s.graph=nn.ModuleList([Graph(C["cdim"]) for _ in range(C["graph_layers"])])
        s.mp=nn.Linear(C["cdim"],C["fusion"])
        D=(0 if AB["strict"] else C["fusion"])+((NCLIN+C["latent"])*C["cdim"] if AB["nodes"] else 0)
        s.corn,s.reason,s.direct=nn.Linear(D,4),nn.Linear(D,5),nn.Linear(NCLIN,5)
        s.mix=nn.Parameter(torch.zeros(3))
    def forward(s,ep):
        al=torch.sigmoid(s.alpha); hr=al*ep[:,0]+(1-al)*graham_gpu(ep[:,1])
        lr=F.interpolate(hr,(C["lr"],C["lr"]),mode="bilinear",align_corners=False).contiguous(memory_format=torch.channels_last)
        f=s.fusion(s.bb(lr)); B=hr.shape[0]
        if AB["ctx"]:
            for c in s.ctx: f=c(f)
        if AB["zoom"]: f,sal=s.zoom(f,hr)
        else: sal=torch.zeros(B,C["grid"]**2,device=hr.device)
        summ=s.summ(torch.cat([f.mean((2,3)),f.amax((2,3))],1))
        adj=None; cl=None; ml=None; n=None
        if AB["nodes"]:
            n=torch.cat([s.dyn(summ).reshape(-1,NCLIN,C["cdim"]),s.lat(summ).reshape(-1,C["latent"],C["cdim"])],1)
            if AB["ground"]: n,_=s.ground(n,f)
            if AB["graph"]:
                for gg in s.graph: n,adj=gg(n)
            cl=s.pres(n[:,:NCLIN]).squeeze(-1)
            ml=torch.einsum('bnd,bdhw->bnhw',s.mp(n[:,:NCLIN]),f)
            flat=n.flatten(1) if AB["strict"] else torch.cat([summ,n.flatten(1)],1)
        else: flat=summ
        zl,rl=s.corn(flat),s.reason(flat)
        cum=zl.float().sigmoid().cumprod(-1)
        cp=torch.cat([1-cum[:,:1],cum[:,:-1]-cum[:,1:],cum[:,-1:]],-1)
        rp=rl.float().softmax(-1)
        if cl is not None:
            dp=s.direct(cl.float().sigmoid()).softmax(-1); w=s.mix.softmax(0); prob=w[0]*cp+w[1]*rp+w[2]*dp
        else: dp=cp; prob=.5*cp+.5*rp
        return dict(prob=prob,corn_prob=cp,reason_prob=rp,direct_prob=dp,corn_logits=zl,
                    concept_logits=cl,h0=n,saliency=sal,adjacency=adj,mask_logits=ml)
def corn_loss(z,t):
    risk=torch.stack([t[:,k:].sum(-1) for k in range(4)],-1); num=torch.stack([t[:,k+1:].sum(-1) for k in range(4)],-1)
    bce=F.binary_cross_entropy_with_logits(z.float(),num/risk.clamp_min(1e-8),reduction="none")
    return (bce*risk).sum()/risk.sum().clamp_min(1)
def grade_loss(o,b):
    t=b["target"].float(); v=t.sum(-1)>0
    if not v.any(): return torch.zeros((),device=t.device,requires_grad=True)
    ce=lambda p: -(t[v]*p[v].float().clamp_min(1e-7).log()).sum(-1).mean()
    g=corn_loss(o["corn_logits"][v],t[v])+.5*ce(o["reason_prob"])+.3*ce(o["prob"])
    if AB["nodes"]: g=g+.5*ce(o["direct_prob"])
    p,q=o["corn_prob"][v].float().clamp_min(1e-7),o["reason_prob"][v].float().clamp_min(1e-7)
    return g+.125*((p*(p.log()-q.log())).sum(-1)+(q*(q.log()-p.log())).sum(-1)).mean()
def aux_loss(o,b):
    lab=b["concept"][:,KEEP]; val=b["mask_valid"][:,KEEP]; mk=b["masks"][:,KEEP]
    tot=torch.zeros((),device=lab.device); kc=lab>=0
    if kc.any(): tot=tot+F.binary_cross_entropy_with_logits(o["concept_logits"][kc],lab[kc].clamp_min(0))
    km=val>0
    if km.any() and o["mask_logits"] is not None:
        lg=F.interpolate(o["mask_logits"],size=mk.shape[-2:],mode="bilinear",align_corners=False)
        tgt=mk; wm=km[:,:,None,None].float()
        pos=tgt*wm; neg=(1-tgt)*wm
        pw=(neg.sum((2,3),keepdim=True)/pos.sum((2,3),keepdim=True).clamp_min(1.)).clamp(1.,30.)
        bce=F.binary_cross_entropy_with_logits(lg.float(),tgt,pos_weight=pw,reduction="none")
        bce=(bce*wm).sum()/(wm.sum()*lg.shape[-1]*lg.shape[-2])
        pr=lg.float().sigmoid()
        num=2*(pr*tgt*wm).sum((0,2,3))+1.0
        den=(wm*(pr+tgt)).sum((0,2,3))+1.0
        has=km.float().sum(0)>0
        dice=((1-num/den)*has).sum()/has.float().sum().clamp_min(1)
        tot=tot+bce+2.*dice
        u=F.adaptive_max_pool2d(mk.max(1,keepdim=True)[0],C["grid"]).flatten(1)
        tot=tot+F.binary_cross_entropy_with_logits(o["saliency"].float(),u,pos_weight=torch.tensor(4.,device=lab.device))
    h=o["h0"].float(); cl,la=h[:,:NCLIN],h[:,NCLIN:]
    cl=F.normalize(cl-cl.mean(-1,keepdim=True),dim=-1); la=F.normalize(la-la.mean(-1,keepdim=True),dim=-1)
    return tot+.05*torch.einsum("bld,bcd->blc",la,cl).square().mean()
class Curr:
    def __init__(s,T,W,S,m,f): s.T,s.W,s.S,s.m,s.f=T,W,S,m,f
    def __call__(s,u):
        u=max(0,int(u))
        if u<s.W: b=(u+1)/max(1,s.W)
        else:
            den=max(1,s.T-s.W-1); pr=min(1.,max(0.,(u-s.W)/den)); b=s.f+(1-s.f)*.5*(1+math.cos(math.pi*pr))
        return b*(s.m if u>=s.S else 1.)
def early(bb):
    m=[getattr(bb,k) for k in ("stem_0","stem_1","stem") if isinstance(getattr(bb,k,None),nn.Module)]
    st=getattr(bb,"stages",None)
    if st is not None and len(st)>=2: m+=[st[0],st[1]]
    else: m+=[getattr(bb,k) for k in ("stages_0","stages_1") if getattr(bb,k,None) is not None]
    return m
def cam_layer(m): return m.fusion

# ---------- FIGURES / GRAD-CAM++ / EXTERNAL ----------
@torch.no_grad()
def epoch_figure(model,d,tag,path,k=4):
    model.eval(); ds=PDS(d,False); idx=np.linspace(0,len(ds)-1,min(k,len(ds))).astype(int)
    fig,ax=plt.subplots(len(idx),4,figsize=(16,4*len(idx)))
    if len(idx)==1: ax=ax[None]
    for r,i in enumerate(idx):
        it=ds[int(i)]; x=it["hr"].unsqueeze(0).to(DEV); o=model(x)
        img=(x[0,0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8); H=img.shape[0]; G=C["grid"]
        sm=F.interpolate(o["saliency"][0].float().reshape(1,1,G,G),(H,H),mode="bilinear",align_corners=False)[0,0].cpu().numpy()
        sm=(sm-sm.min())/(sm.max()-sm.min()+1e-8)
        hm=cv2.applyColorMap(np.uint8(255*sm),cv2.COLORMAP_JET)
        ov=cv2.addWeighted(img,.6,cv2.cvtColor(hm,cv2.COLOR_BGR2RGB),.4,0)
        pr=o["prob"][0].float().cpu().numpy()
        ax[r,0].imshow(img); ax[r,0].set_title(f"true {int(it['grade'])} | pred {int(pr.argmax())}"); ax[r,0].axis("off")
        ax[r,1].imshow(ov); ax[r,1].set_title("intrinsic saliency"); ax[r,1].axis("off")
        if o["mask_logits"] is not None:
            ml=F.interpolate(o["mask_logits"],(H,H),mode="bilinear",align_corners=False)[0].float().sigmoid().cpu().numpy()
            j=KN.index("EX") if "EX" in KN else 0
            cs_,_=cv2.findContours((ml[j]>.5).astype(np.uint8)*255,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            vis=img.copy(); cv2.drawContours(vis,cs_,-1,(0,255,0),3)
            ax[r,2].imshow(vis); ax[r,2].set_title(f"{KN[j]} mask"); ax[r,2].axis("off")
            ax[r,3].barh(KN,o["concept_logits"][0].float().sigmoid().cpu().numpy(),color="steelblue")
            ax[r,3].set_xlim(0,1); ax[r,3].invert_yaxis(); ax[r,3].set_title("concept probs")
        else: ax[r,2].axis("off"); ax[r,3].axis("off")
    fig.suptitle(tag,fontsize=14); fig.tight_layout(); fig.savefig(path,dpi=85,bbox_inches="tight"); plt.close(fig)
def gradcampp_run(model,d,run,n=6):
    gd=run/"gradcampp"; gd.mkdir(exist_ok=True)
    prev=[p.requires_grad for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(True)
    store={}; layer=cam_layer(model)
    h1=layer.register_forward_hook(lambda m,i,o: store.__setitem__("a",o))
    h2=layer.register_full_backward_hook(lambda m,gi,go: store.__setitem__("g",go[0]))
    ds=PDS(d,False); sel=np.linspace(0,len(ds)-1,min(n,len(ds))).astype(int); rows=[]; okc=0
    try:
        model.eval()
        for i in sel:
            it=ds[int(i)]; x=it["hr"].unsqueeze(0).to(DEV)
            model.zero_grad(set_to_none=True); store.clear()
            out=model(x)["prob"]; cls=int(out.argmax(1))
            torch.log(out[0,cls]+1e-9).backward()
            if "a" not in store or "g" not in store: raise RuntimeError("hook did not fire")
            acts=store["a"][0].detach().float(); g1=store["g"][0].detach().float()
            g2=g1*g1; g3=g2*g1; ssum=g1.sum((1,2),keepdim=True)
            den=2*g2+g3*ssum
            alpha=torch.where(den.abs()>1e-12,g2/(den+1e-12),torch.zeros_like(g2))
            w=(alpha*F.relu(g1)).sum((1,2))
            cam=F.relu((w[:,None,None]*acts).sum(0)); cam=(cam/(cam.max()+1e-8)).cpu().numpy()
            img=(x[0,0].permute(1,2,0).detach().cpu().numpy()*255).clip(0,255).astype(np.uint8)
            cm=cv2.resize(cam,(img.shape[1],img.shape[0]))
            hm=cv2.applyColorMap(np.uint8(255*cm),cv2.COLORMAP_JET)
            ov=cv2.addWeighted(img,.6,cv2.cvtColor(hm,cv2.COLOR_BGR2RGB),.4,0)
            rows.append(np.hstack([img,cv2.cvtColor(hm,cv2.COLOR_BGR2RGB),ov]))
            cv2.imwrite(str(gd/f"{int(i)}_true{int(it['grade'])}_pred{cls}.jpg"),cv2.cvtColor(ov,cv2.COLOR_RGB2BGR))
            okc+=1
        if rows: cv2.imwrite(str(run/"gradcampp_summary.jpg"),cv2.cvtColor(np.vstack(rows),cv2.COLOR_RGB2BGR))
        print(f"Grad-CAM++ saved {okc} -> {run/'gradcampp_summary.jpg'}",flush=True)
    except Exception as e: print("Grad-CAM++ FAILED:",type(e).__name__,e,flush=True)
    finally:
        h1.remove(); h2.remove()
        for p,f_ in zip(model.parameters(),prev): p.requires_grad_(f_)
        model.zero_grad(set_to_none=True)
def _img_index(root):
    idx={}
    for p in Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() in (".jpg",".jpeg",".png",".tif",".tiff",".bmp"):
            idx[p.stem.lower()]=p; idx[p.name.lower()]=p
    return idx
def _pick(cols,keys):
    for k in keys:
        for c in cols:
            if k in str(c).lower().replace(" ","_"): return c
    return None
class EDS(Dataset):
    def __init__(s,rows): s.rows=rows
    def __len__(s): return len(s.rows)
    def __getitem__(s,i):
        path,gg=s.rows[i]; bgr=cv2.imread(str(path))
        if bgr is None: return dict(hr=torch.zeros(2,3,C["hr"],C["hr"]),grade=torch.tensor(0),ok=False)
        cl,raw,_=ends(bgr); hr=np.stack([cl,raw]).transpose(0,3,1,2).copy()
        return dict(hr=torch.from_numpy(hr).float()/255,grade=torch.tensor(int(gg)),ok=True)
def ext_collate(batch):
    batch=[b for b in batch if bool(b["ok"])]
    if not batch: return None
    return {k:torch.stack([b[k] for b in batch]) for k in ("hr","grade")}

# ---------- PREFLIGHT ----------
banner("PREFLIGHT")
for V in ABs:
    set_variant(V)
    m=Net().to(DEV); m.eval()
    x=torch.zeros(1,2,3,C["hr"],C["hr"],device=DEV)
    with torch.no_grad(): o=m(x)
    assert o["prob"].shape==(1,5)
    if AB["nodes"]:
        assert o["concept_logits"].shape==(1,NCLIN) and o["mask_logits"].shape[1]==NCLIN
    tg=torch.zeros(1,5,device=DEV); tg[0,2]=1.
    gg_=grade_loss(o,dict(target=tg,grade=torch.tensor([2],device=DEV)))
    assert torch.isfinite(gg_)
    del m; torch.cuda.empty_cache(); print(f"  {V:11s} forward+loss OK",flush=True)
set_variant(VARIANT)
b=next(iter(DataLoader(ADS(aux_tr[:8],True),batch_size=2,shuffle=False,num_workers=0)))
print("ADS batch:",{k:tuple(v.shape) for k,v in b.items()},flush=True)
b2=next(iter(ploader(P["train"],True,True,0)))
print("PDS batch:",{k:tuple(v.shape) for k,v in b2.items()},flush=True)
print("PREFLIGHT PASSED",flush=True)

# ---------- TRAIN ----------
banner("TRAINING")
alo=DataLoader(ADS(aux_tr,True),batch_size=C["aux_batch"],shuffle=True,num_workers=C["workers"],pin_memory=True)
model=Net().to(DEV).to(memory_format=torch.channels_last)
ema=copy.deepcopy(model).eval().requires_grad_(False)
ntr=lambda: sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params={sum(p.numel() for p in model.parameters())/1e6:.2f}M trainable={ntr()/1e6:.2f}M",flush=True)
opt=torch.optim.AdamW([{"params":[p for n,p in model.named_parameters() if n.startswith("bb.")],"lr":C["head_lr"]*C["lr_ratio"]},
                       {"params":[p for n,p in model.named_parameters() if not n.startswith("bb.")],"lr":C["head_lr"]}],
                      weight_decay=C["wd"],betas=(.9,.999),eps=1e-8)
upe=math.ceil(len(P["train"])/(C["batch"]*C["accum"])); dec=math.exp(-1./(C["tau"]*upe))
sch=torch.optim.lr_scheduler.LambdaLR(opt,Curr(C["epochs"]*upe,max(1,round(C["warmup"]*C["epochs"]*upe)),C["stage1"]*upe,C["stage2_mult"],C["cos_floor"]))
sc_=amp_scaler(DEV=="cuda")
(RUN/"config.json").write_text(json.dumps(dict(C,AB=AB,TAG=TAG,VARIANT=VARIANT,FOLD=FOLD,ema=dec,upe=upe,
    groups_file=str(GRP),folds_file=str(FJ)),indent=2))
@torch.no_grad()
def predict(d,tta=1):
    ema.eval(); ps,cs,ys=[],[],[]; asum=None; cnt=0
    for b in ploader(d,False,False,0):
        x=b["hr"].to(DEV)
        vs=[x,x.flip(-1),x.flip(-2),x.flip(-1).flip(-2),torch.rot90(x,1,(-2,-1)),torch.rot90(x,3,(-2,-1))][:tta]
        outs=[ema(v) for v in vs]
        ps.append(torch.stack([q["prob"].float() for q in outs]).mean(0).cpu().numpy())
        if outs[0]["concept_logits"] is not None:
            cs.append(torch.stack([q["concept_logits"].float().sigmoid() for q in outs]).mean(0).cpu().numpy())
        if outs[0]["adjacency"] is not None:
            a=outs[0]["adjacency"].float().sum(0).cpu().numpy(); asum=a if asum is None else asum+a; cnt+=x.shape[0]
        ys.append(b["grade"].numpy())
    return np.concatenate(ps),np.concatenate(ys),(np.concatenate(cs) if cs else None),(asum/max(1,cnt) if asum is not None else None)
def thr_score(sc,y,th):
    p=np.digitize(sc,th); q=qwk_fast(y,p); pen=0.
    for gg in range(5):
        nn_=int((y==gg).sum())
        if nn_>=10: pen+=max(0.,.30-float(((p==gg)&(y==gg)).sum())/nn_)
    return q-.35*pen
def fit_thr(sc,y,init=None):
    th=np.array(init if init is not None else [.5,1.5,2.5,3.5]); bq=thr_score(sc,y,th)
    cand=np.unique(np.quantile(sc,np.linspace(.02,.98,49)))
    for _ in range(3):
        imp=False
        for k in range(4):
            bv,bb=th[k],bq
            for c in cand:
                t=th.copy(); t[k]=c
                if not np.all(np.diff(t)>0): continue
                q=thr_score(sc,y,t)
                if q>bb+1e-9: bb,bv=q,c
            if bv!=th[k]: th[k],bq,imp=bv,bb,True
        if not imp: break
    return th
best=dict(nll=1e9,qwk=-1,ep=-1); log=[]; ait=iter(alo)
if DEV=="cuda": torch.cuda.reset_peak_memory_stats()
T0=time.perf_counter()
for ep in range(C["epochs"]):
    if ep==C["stage1"]:
        b0=ntr()
        for m in early(model.bb):
            for p in m.parameters(): p.requires_grad=False
        assert ntr()<b0
        print(f"[stage2] {b0/1e6:.2f}M -> {ntr()/1e6:.2f}M",flush=True)
    model.train(); opt.zero_grad(set_to_none=True)
    ld=ploader(P["train"],True,ep<C["stage1"],ep); n=len(ld)
    rl_=0.; ra_=0.; seen=0; trP,trY=[],[]
    bar=tqdm(ld,total=n,desc=f"ep{ep:02d}",leave=False,dynamic_ncols=True)
    for st,b in enumerate(bar):
        b={k:(v.to(DEV,non_blocking=True) if torch.is_tensor(v) else v) for k,v in b.items()}
        if ep<C["stage1"]: b=mixup(b,C["mixup_alpha"])
        win=min(C["accum"],n-(st//C["accum"])*C["accum"])
        with amp_ctx(DEV=="cuda"): o=model(b["hr"])
        with amp_ctx(False): gl=grade_loss(o,b)
        with torch.no_grad():
            pr=o["prob"].argmax(1); trP.extend(pr.cpu().tolist()); trY.extend(b["grade"].cpu().tolist())
            bs_=len(pr); ab_=float((pr==b["grade"]).float().mean())
            ra_=(ra_*seen+ab_*bs_)/(seen+bs_); seen+=bs_
        av=0.
        if AUX_ON and st%C["aux_every"]==0:
            try: ab=next(ait)
            except StopIteration: ait=iter(alo); ab=next(ait)
            ab={k:v.to(DEV,non_blocking=True) for k,v in ab.items()}
            with amp_ctx(DEV=="cuda"): ao=model(ab["hr"])
            with amp_ctx(False): al=aux_loss(ao,ab); av=float(al.detach())
            gl=gl+C["aux_w"]*C["aux_every"]*al
        sc_.scale(gl/win).backward(); rl_=(rl_*st+float(gl.detach()))/(st+1)
        bar.set_postfix(acc=f"{ra_:.3f}",loss=f"{rl_:.3f}",aux=f"{av:.2f}",lr=f"{opt.param_groups[-1]['lr']:.1e}")
        if (st+1)%C["accum"]==0 or (st+1)==n:
            sc_.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),C["clip"])
            pv=sc_.get_scale(); sc_.step(opt); sc_.update()
            if sc_.get_scale()>=pv:
                sch.step()
                with torch.no_grad():
                    for e,p in zip(ema.parameters(),model.parameters()): e.lerp_(p.detach(),1-dec)
                    for e,p in zip(ema.buffers(),model.buffers()): e.copy_(p)
            opt.zero_grad(set_to_none=True)
    bar.close()
    TM=full_metrics(trY,trP); tp,ty,_,_=predict(P["tune"],1); VM=full_metrics(ty,tp.argmax(1),tp)
    qr=qwk_fast(ty,np.clip(np.rint(tp@np.arange(5)),0,4))
    mem=torch.cuda.max_memory_allocated()/2**30 if DEV=="cuda" else 0.
    el=time.perf_counter()-T0; eta=(el/(ep+1))*(C["epochs"]-ep-1)/3600
    rec=dict(epoch=ep,seconds=el,peak_gib=mem,lr=opt.param_groups[-1]["lr"],tune_qwk_round=qr,train=TM,val=VM)
    log.append(rec)
    print_block(f"EP {ep:02d} VAL",VM,extra=f" | nll {VM['nll']:.4f} ece {VM['ece']:.4f} qwk_round {qr:.4f} | tr_acc {TM['acc']:.3f} tr_qwk {TM['qwk']:.3f}"
              f" | lr {rec['lr']:.2e} mem {mem:.1f}G {el/60:.1f}m ETA {eta:.1f}h best_nll {best['nll']:.4f}@{best['ep']}")
    with (RUN/"epochs.jsonl").open("a") as f: f.write(json.dumps(rec)+"\n")
    epoch_figure(ema,P["tune"],f"epoch {ep} (val)",FIGS/f"ep{ep:02d}.png",C["epoch_figs"])
    if VM["nll"]<best["nll"]-1e-6 or (abs(VM["nll"]-best["nll"])<=1e-6 and VM["qwk"]>best["qwk"]+1e-6):
        best=dict(nll=VM["nll"],qwk=VM["qwk"],ep=ep); torch.save(ema.state_dict(),RUN/"best_ema.pt")
    if mem>C["max_gib"]: raise RuntimeError(f"peak {mem:.2f} GiB > {C['max_gib']} -> batch=2,accum=8")
torch.save(ema.state_dict(),RUN/"last_ema.pt")

# ---------- CALIBRATION (T FIRST) + TEST ----------
banner("CALIBRATION + TEST (once)")
ema.load_state_dict(torch.load(RUN/"best_ema.pt",map_location=DEV,weights_only=True))
bp,by,_,_=predict(P["cal_b"],1); bl=np.log(bp.clip(1e-9)); Ts=np.linspace(.5,3.,51); nls=[]
for t in Ts:
    lp=bl/t; lp=lp-np.log(np.exp(lp).sum(1,keepdims=True)); nls.append(-lp[np.arange(len(by)),by].mean())
T=float(Ts[int(np.argmin(nls))])
def tsc(p):
    lp=np.log(p.clip(1e-9))/T; return np.exp(lp-np.log(np.exp(lp).sum(1,keepdims=True)))
ap,ay,_,_=predict(P["cal_a"],C["tta"]); apT=tsc(ap); asc=apT@np.arange(5)
rng=np.random.default_rng(SEED); ths=[]
cnt=np.bincount(P["train"].diagnosis.values,minlength=5).astype(float); prior=np.cumsum(cnt/cnt.sum())[:-1]
init=np.quantile(asc,prior)
for _ in range(C["thr_boot"]):
    i=rng.integers(0,len(ay),len(ay))
    if len(np.unique(ay[i]))<2: continue
    ths.append(fit_thr(asc[i],ay[i],init))
thr=np.sort(np.mean(ths,0)) if ths else fit_thr(asc,ay,init)
for k in range(1,4):
    if thr[k]<=thr[k-1]: thr[k]=thr[k-1]+1e-3
ref_thr,bs=3.5,-1
for t in np.unique(asc):
    se=((asc>=t)&(ay>=2)).sum()/max(1,(ay>=2).sum()); sp=((asc<t)&(ay<2)).sum()/max(1,(ay<2).sum())
    if se>=.95 and sp>bs: bs,ref_thr=sp,float(t)
bpT=tsc(bp)
def aps_sc(p,y):
    o=np.argsort(-p,1); sp=np.take_along_axis(p,o,1); cu=sp.cumsum(1); u=rng.uniform(size=len(y))
    pos=[np.where(o[i]==y[i])[0][0] for i in range(len(y))]
    return np.array([(cu[i,pos[i]-1] if pos[i]>0 else 0.)+u[i]*sp[i,pos[i]] for i in range(len(y))])
def aps_q(s,a):
    n=len(s); k=min(n,int(math.ceil((n+1)*(1-a)))); return float(np.sort(s)[k-1])
def aps_set(p,q):
    o=np.argsort(-p,1); cu=np.take_along_axis(p,o,1).cumsum(1); out=np.zeros_like(p,bool)
    for i in range(len(p)):
        k=min(p.shape[1],int(np.searchsorted(cu[i],q))+1); out[i,o[i,:k]]=True
    return out
qs={a:aps_q(aps_sc(bpT,by),a) for a in (.05,.10)}
print(f"T={T:.3f} thr={np.round(thr,3).tolist()} ref_thr={ref_thr:.3f}",flush=True)
vp,vy,vconc,vadj=predict(P["test"],C["tta"]); vpT=tsc(vp)
vsc=vpT@np.arange(5); vpred=np.digitize(vsc,thr)
FM=full_metrics(vy,vpred,vpT)
def boot(y,p,B=2000):
    r=np.random.default_rng(SEED); v=[]
    for _ in range(B):
        i=r.integers(0,len(y),len(y))
        if len(np.unique(y[i]))<2: continue
        v.append(cohen_kappa_score(y[i],p[i],weights="quadratic"))
    return np.percentile(v,[2.5,97.5]).tolist()
sets={f"{int((1-a)*100)}%":aps_set(vpT,q) for a,q in qs.items()}; rp=vsc>=ref_thr
refp=(vy>=2).astype(float); refpr=vpT[:,2:].sum(1); bins=np.linspace(0,1,11)
ref_ece=float(sum(((refpr>=bins[i])&(refpr<bins[i+1])).mean()*
                  abs(refp[(refpr>=bins[i])&(refpr<bins[i+1])].mean()-refpr[(refpr>=bins[i])&(refpr<bins[i+1])].mean())
                  for i in range(10) if ((refpr>=bins[i])&(refpr<bins[i+1])).sum()>0))
res=dict(tag=TAG,variant=VARIANT,fold=FOLD,best_epoch=best["ep"],T=T,thresholds=thr.tolist(),ref_thr=ref_thr,
         test=FM,ci95=boot(vy,vpred,C["boot"]),qwk_argmax=qwk_fast(vy,vpT.argmax(1)),
         mae=float(np.abs(vsc-vy).mean()),
         referable_sens=float((rp&(vy>=2)).sum()/max(1,(vy>=2).sum())),
         referable_spec=float(((~rp)&(vy<2)).sum()/max(1,(vy<2).sum())),referable_ece=ref_ece,
         conformal={k:dict(coverage=float(np.mean([sets[k][i,vy[i]] for i in range(len(vy))])),
                           set_size=float(sets[k].sum(1).mean())) for k in sets},
         mix_weights=ema.mix.softmax(0).detach().cpu().numpy().tolist() if AB["nodes"] else None,
         gates=dict(alpha=float(torch.sigmoid(ema.alpha).detach()),zoom=float(ema.zoom.gate.detach()),
                    ctx=[float(c.g.detach()) for c in ema.ctx]),
         split_sizes={k:len(v) for k,v in P.items()},
         split_dist={k:{int(gg):int((P[k].diagnosis==gg).sum()) for gg in range(5)} for k in P})
print_block("TEST",FM,extra=f" | CI {res['ci95']} mae {res['mae']:.3f}")
print(f"referable sens {res['referable_sens']:.3f} spec {res['referable_spec']:.3f} refECE {ref_ece:.4f}",flush=True)
print("conformal:",json.dumps(res["conformal"]),flush=True)
print("mix:",res["mix_weights"]," gates:",res["gates"],flush=True)
if vadj is not None: np.save(RUN/"adjacency.npy",vadj); res["adjacency_clinical"]=vadj[:NCLIN,:NCLIN].tolist()
if vconc is not None:
    with torch.no_grad():
        ct=torch.from_numpy(vconc).to(DEV); base=torch.softmax(ema.direct(ct),-1).cpu().numpy(); inter={}
        for j,cn in enumerate(KN):
            row={}
            for v_ in (0.,1.):
                c=ct.clone(); c[:,j]=v_; p=torch.softmax(ema.direct(c),-1).cpu().numpy()
                row[f"set{int(v_)}"]=dict(dgrade=float(np.abs(p@np.arange(5)-base@np.arange(5)).mean()),
                                          qwk=qwk_fast(vy,np.digitize(p@np.arange(5),thr)))
            inter[cn]=row
        res["concept_sensitivity_direct_head"]=inter
np.savez_compressed(RUN/"test_preds.npz",prob=vpT,y=vy,pred=vpred,score=vsc)

# ---------- GRAD-CAM++ / FAITHFULNESS / SANITY ----------
banner("GRAD-CAM++ + FAITHFULNESS")
gradcampp_run(ema,P["test"],RUN,C["cam_images"])
if AB["zoom"]:
    @torch.no_grad()
    def faith(nper=6,steps=10):
        ds=PDS(P["test"],False); sel=[]
        for gg in range(5): sel+=list(np.where(P["test"].diagnosis.values==gg)[0][:nper])
        D=np.zeros(steps+1); I=np.zeros(steps+1); nb=0
        for i in sel:
            it=ds[int(i)]; x=it["hr"].unsqueeze(0).to(DEV); o=ema(x)
            cls=int(o["prob"][0].argmax()); G=C["grid"]
            sm=F.interpolate(o["saliency"][0].reshape(1,1,G,G).float(),(C["hr"],C["hr"]),mode="bilinear",align_corners=False)[0,0]
            order=torch.argsort(sm.flatten(),descending=True); npix=sm.numel()
            for k in range(steps+1):
                nm=int(k/steps*npix); m=torch.ones(npix,device=DEV)
                if nm>0: m[order[:nm]]=0
                m=m.reshape(1,1,C["hr"],C["hr"])
                D[k]+=float(ema(x*m)["prob"][0,cls]); I[k]+=float(ema(x*(1-m))["prob"][0,cls])
            nb+=1
        return (D/nb).tolist(),(I/nb).tolist(),float(_trapz(D/nb,dx=1/steps)),float(_trapz(I/nb,dx=1/steps))
    dl,ins,da,ia=faith(); san=None
    try:
        m2=copy.deepcopy(ema)
        for mod in [m2.zoom,m2.mp]:
            for p in mod.parameters():
                if p.dim()>1: nn.init.kaiming_uniform_(p,a=math.sqrt(5))
                else: nn.init.zeros_(p)
        m2.eval(); ds=PDS(P["test"],False); a_,b_=[],[]
        with torch.no_grad():
            for i in range(min(30,len(ds))):
                x=ds[i]["hr"].unsqueeze(0).to(DEV)
                s1=ema(x)["saliency"][0].float().cpu().numpy(); s2=m2(x)["saliency"][0].float().cpu().numpy()
                a_.append(np.argsort(np.argsort(s1))); b_.append(np.argsort(np.argsort(s2)))
        san=float(np.corrcoef(np.concatenate(a_),np.concatenate(b_))[0,1]); del m2; torch.cuda.empty_cache()
    except Exception as e: print("sanity skipped:",e,flush=True)
    res["faithfulness"]=dict(deletion=dl,insertion=ins,deletion_auc=da,insertion_auc=ia,sanity_rank_corr=san)
    print(f"deletion AUC {da:.3f} insertion AUC {ia:.3f} sanity {san}",flush=True)

# ---------- LOCALIZATION ----------
banner("LOCALIZATION (held-out expert masks)")
@torch.no_grad()
def seg_eval(rows):
    ema.eval(); ds=DataLoader(ADS(rows,False),batch_size=1,shuffle=False,num_workers=2)
    it={c:0. for c in KN}; un={c:0. for c in KN}; pP={c:[] for c in KN}; pT={c:[] for c in KN}; pg={c:{} for c in KN}
    for b in ds:
        o=ema(b["hr"].to(DEV))
        if o["mask_logits"] is None: break
        ml=F.interpolate(o["mask_logits"],(MASKSZ,MASKSZ),mode="bilinear",align_corners=False)
        pm=ml[0].float().sigmoid().cpu().numpy(); pc=o["concept_logits"][0].float().sigmoid().cpu().numpy()
        gt=b["masks"][0,KEEP].numpy(); kv=b["mask_valid"][0,KEEP].numpy()>0; gg=int(b["grade"][0])
        for j,c in enumerate(KN):
            if not kv[j]: continue
            p=(pm[j]>.5).astype(float); q=gt[j].astype(float)
            it[c]+=(p*q).sum(); un[c]+=p.sum()+q.sum(); pP[c].append(pc[j]); pT[c].append(float(q.sum()>0))
            if gg>=0:
                dd=pg[c].setdefault(gg,[0.,0.]); dd[0]+=(p*q).sum(); dd[1]+=p.sum()+q.sum()
    return {c:dict(dice=round(2*it[c]/un[c],4) if un[c]>0 else None,
                   presence_auc=float(roc_auc_score(pT[c],pP[c])) if (len(pT[c])>5 and len(set(pT[c]))>1) else None,
                   n=len(pT[c]),dice_per_grade={gg:round(2*v[0]/v[1],4) if v[1]>0 else None for gg,v in pg[c].items()}) for c in KN}
sg=seg_eval(aux_ho); res["localization"]=sg
for c,v in sg.items(): print(f"  {c:5s} dice={v['dice']} auc={v['presence_auc']} n={v['n']}",flush=True)

# ---------- EXTERNAL ----------
@torch.no_grad()
def external_eval(name,csv,imgdir):
    banner(f"EXTERNAL zero-shot — {name}")
    if not Path(csv).exists() or not Path(imgdir).exists():
        print(f"[SKIP] {name}: path missing",flush=True); return None
    dfx=pd.read_csv(csv); print("cols:",dfx.columns.tolist(),flush=True)
    idc=_pick(dfx.columns,["image_id","id_code","image","filename","file","name","id"])
    gc=_pick(dfx.columns,["adjudicated_dr_grade","dr_grade","retinopathy_grade","diagnosis","grade","label","class","level"])
    if idc is None or gc is None: print("[SKIP] columns not found",flush=True); return None
    gradable=_pick(dfx.columns,["gradable"])
    if gradable is not None and gradable!=gc:
        b_=len(dfx); dfx=dfx[dfx[gradable]!=0]; print(f"dropped ungradable {b_-len(dfx)}",flush=True)
    dfx[gc]=pd.to_numeric(dfx[gc],errors="coerce"); dfx=dfx[dfx[gc].notna()]
    b_=len(dfx); dfx=dfx[(dfx[gc]>=0)&(dfx[gc]<=4)]; print(f"dropped out-of-range {b_-len(dfx)}",flush=True)
    dist_report(name,dfx.rename(columns={gc:"diagnosis"}))
    idx=_img_index(imgdir); rows=[]; miss=0
    for _,r in dfx.iterrows():
        key=str(r[idc]).strip().lower(); p=idx.get(key) or idx.get(Path(key).stem)
        if p is None: miss+=1; continue
        rows.append((p,int(r[gc])))
    print(f"matched {len(rows)} missing {miss}",flush=True)
    if not rows: return None
    ld=DataLoader(EDS(rows),batch_size=C["eval_batch"],shuffle=False,num_workers=C["workers"],
                  pin_memory=True,collate_fn=ext_collate)
    ema.eval(); ps,ys=[],[]; dropped=0
    for b in ld:
        if b is None: dropped+=1; continue
        x=b["hr"].to(DEV); vs=[x,x.flip(-1),x.flip(-2),x.flip(-1).flip(-2)]
        ps.append(torch.stack([ema(v)["prob"].float() for v in vs]).mean(0).cpu().numpy()); ys.append(b["grade"].numpy())
    if dropped: print(f"[WARN] {dropped} unreadable batches skipped",flush=True)
    vp_=np.concatenate(ps); vy_=np.concatenate(ys)
    vpT_=tsc(vp_); vsc_=vpT_@np.arange(5); vpred_=np.digitize(vsc_,thr)
    M=full_metrics(vy_,vpred_,vpT_); rp_=vsc_>=ref_thr
    sets_={f"{int((1-a)*100)}%":aps_set(vpT_,q) for a,q in qs.items()}
    r=np.random.default_rng(SEED); bv=[]
    for _ in range(C["boot"]):
        i=r.integers(0,len(vy_),len(vy_))
        if len(np.unique(vy_[i]))<2: continue
        bv.append(cohen_kappa_score(vy_[i],vpred_[i],weights="quadratic"))
    out=dict(name=name,n=len(vy_),labels=sorted(set(vy_.tolist())),metrics=M,
             ci95=np.percentile(bv,[2.5,97.5]).tolist() if bv else None,mae=float(np.abs(vsc_-vy_).mean()),
             referable_sens=float((rp_&(vy_>=2)).sum()/max(1,(vy_>=2).sum())),
             referable_spec=float(((~rp_)&(vy_<2)).sum()/max(1,(vy_<2).sum())),
             conformal_transfer={k:dict(coverage=float(np.mean([sets_[k][i,vy_[i]] for i in range(len(vy_))])),
                                        set_size=float(sets_[k].sum(1).mean())) for k in sets_})
    print_block(f"[{name}] EXTERNAL",M,extra=f" | CI {out['ci95']} mae {out['mae']:.3f}")
    print(f"referable sens {out['referable_sens']:.3f} spec {out['referable_spec']:.3f}",flush=True)
    print("conformal TRANSFER:",json.dumps(out["conformal_transfer"]),flush=True)
    np.savez_compressed(RUN/f"external_{name.replace('-','').lower()}.npz",prob=vpT_,y=vy_,pred=vpred_,score=vsc_)
    return out
res["external"]={}
for nm,csv_,dir_ in [("Messidor-2",M2_CSV,M2_DIR),("DDR",DDR_CSV,DDR_DIR)]:
    res["external"][nm]=external_eval(nm,csv_,dir_)

# ---------- SAVE ----------
res["minutes"]=(time.perf_counter()-T0)/60; res["peak_gib"]=max(r["peak_gib"] for r in log)
(RUN/"metrics.json").write_text(json.dumps(res,indent=2,default=str))
banner("FINAL SUMMARY")
print(f"ARM {TAG} FOLD {FOLD} | QWK {FM['qwk']:.4f} CI {res['ci95']} | ACC {FM['acc']:.4f} BAL {FM['bal_acc']:.4f} | gates {res['gates']}")
print(f"minutes {res['minutes']:.1f} peak {res['peak_gib']:.1f} GiB | saved {RUN}",flush=True)

dynamo: REAL
import OK: torchvision
import OK: timm

FOLD 0 | ARM grouped | VARIANT full | hr1024 lr384 patch160 grid8 mask128 gate_init1.0
[grouped] clean pool: 3583 images / 3477 groups
POOL         n= 3583 | g0:1801( 50.3%)  g1: 360( 10.0%)  g2: 975( 27.2%)  g3: 172(  4.8%)  g4: 275(  7.7%)
[CHECK] group-confined: 0 duplicate-groups cross any split
--- CLASS DISTRIBUTION ---
TRAIN        n= 1788 | g0: 889( 49.7%)  g1: 167(  9.3%)  g2: 504( 28.2%)  g3:  87(  4.9%)  g4: 141(  7.9%)
TUNE         n=  354 | g0: 190( 53.7%)  g1:  35(  9.9%)  g2:  86( 24.3%)  g3:  20(  5.6%)  g4:  23(  6.5%)
CAL_A        n=  361 | g0: 179( 49.6%)  g1:  41( 11.4%)  g2:  92( 25.5%)  g3:  17(  4.7%)  g4:  32(  8.9%)
CAL_B        n=  361 | g0: 175( 48.5%)  g1:  42( 11.6%)  g2: 100( 27.7%)  g3:  20(  5.5%)  g4:  24(  6.6%)
TEST         n=  719 | g0: 368( 51.2%)  g1:  75( 10.4%)  g2: 193( 26.8%)  g3:  28(  3.9%)  g4:  55(  7.6%)
built aux: 1923
aux train=1527 holdout=396

PREFLIGHT


model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            

  full        forward+loss OK
  strict      forward+loss OK
  graph_free  forward+loss OK
  baseline    forward+loss OK
  no_vb_irma  forward+loss OK
ADS batch: {'hr': (2, 2, 3, 1024, 1024), 'masks': (2, 8, 128, 128), 'mask_valid': (2, 8), 'concept': (2, 8), 'grade': (2,)}
PDS batch: {'hr': (4, 2, 3, 1024, 1024), 'target': (4, 5), 'grade': (4,)}
PREFLIGHT PASSED

TRAINING
params=18.87M trainable=18.87M


ep00:   0%|          | 0/447 [00:00<?, ?it/s]